In [2]:
import easyocr
import cv2 as cv
import numpy as np
import os
import re

# code from https://stackoverflow.com/questions/76723117/divide-an-image-into-tiles-based-on-text-structure-in-python-opencv

def to_rect(box):
    xlist = [p[0] for p in box] #0 = x coordinates
    ylist = [p[1] for p in box] #1 = y coordinates
    return {
        "left": min(xlist),
        "right": max(xlist),
        "top": min(ylist),
        "bottom": max(ylist),
        "height": max(ylist) - min(ylist),
        "width": max(xlist) - min(xlist),
        "center_y": (min(ylist) + max(ylist)) / 2,
        "center_x": (min(xlist) + max(xlist)) / 2
    }


def structure_data(result):
    structured = []
    for bbox, text in result:
        structured.append({
            "rect": to_rect(bbox),
            "text": text
        })

    return structured   

def tokenize_alpha(text):
    # keep only letters and spaces
    cleaned = re.sub(r'[^A-Za-z\s]', '', text)
    # collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

LIST_OF_NON_MENU_KEYWORDS = ['.com', 'london', 'lunch', 'dinner', 'starter', 'main', 'dessert', 'cafe', 'please', 'service charge'] # expand l8r

PRICE_PATTERN = re.compile(r"""
    [£$€EeLlIi{YS]   # currency-like symbol
    \s*
    \d+               # digits
    (\.\d{1,2})?      # optional decimal
""", re.VERBOSE)

def reject_non_menu_items(text):
    if(len(text) > 2):
        if not any(w in text.lower() for w in LIST_OF_NON_MENU_KEYWORDS):
            # only return if the text contains no forbidden key substrings
            return True
    return False

def clean_menu_item(text):
    #print("items before clean", text)
    # remove allergen codes like (G/D/N)
    text = re.sub(r"\(([A-Za-z/]+)\)", "", text)
    #print("text subbed bracket patterns: ", text)

    # remove price-like codes such as E1.99, E3, + E1.99
    text = re.sub(PRICE_PATTERN, "", text)
    #print("text subbed price patterns: ", text)

    # collapse extra spaces created by removals
    text = re.sub(r"\s{2,}", " ", text).strip()
    #print("text subbed extra space: ", text)

    return text

def item_to_text(item):
    return " ".join(b["text"] for b in item)


path = "pariscafe1.jpg"
assert os.path.exists(path)

def run_ocr(path):
    #always a good idea to convert BGR to RGB when using OCR
    img = cv.imread(path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    #read the text
    reader = easyocr.Reader(['en'])
    img_height = img.shape[0]
    y_threshold = img_height * 0.000015
    print(img_height, y_threshold)
    text_data = reader.readtext(img, paragraph=True, x_ths=0.5)     #in order ([box-coords], text, confidence)
    visualize(text_data, img)
    #print(text_data)

    rect_data = structure_data(text_data)

    #print(rect_data)

    items = []

    for para in rect_data:
        items.append(para["text"])

    #Dont bother cleaning non-menu items
    non_menu = [t for t in items if reject_non_menu_items(t)]

    clean_items = [clean_menu_item(t) for t in non_menu]
    clean_items = [t for t in clean_items if t.strip()] #remove empty strings
    print(clean_items)

    return(clean_items)

def visualize(text_data, img):
    viz_img = np.copy(img)
    #visualize
    for data in text_data:
        # box, text
        box, text = data
        top_left, top_right, bottom_right, bottom_left = box

        tl = [int(x) for x in top_left]
        br = [int(x) for x in bottom_right]
        cv.rectangle(viz_img, tl, br, (0, 255, 0), 4)
        cv.putText(viz_img, text, br, cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv.imwrite('viz_with_text.jpg', viz_img)

In [4]:
#from ocr_engine import run_ocr as import_run_ocr

menu_items = run_ocr('the_bach.jpg')

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


3195 0.047925
["just because we can here's some kai from our evening menu; all cooked aver kanaku ~enjoy something a little different vo", 'SUNDAY SPECIALS', 'FROM KANAKU', '12 HOUR SMOKED BRISKET a slice of our 12 hour smoked brisket served with bachmade potato salad and a complimentary sauce of your choice', '12.0', '8 HouR SMOKED BEEF RIB TACOS tender beef short rib on the bone served with six fired tacos, house slaw; peach salsa and bachmade jalapeno & coriander crema', '23.0', '8 HOUR SMOKED PoRK BELLY our pork belly is cooked low and slow and just falls apart none of this undercooked shttyou often get when out in a restaurant: served with apple slaw and bachmade star anise & chilli syrup', 'CHICKEN DRUMETTES five chicken drumettes in our house rub, cooked over fire, tossed in sriracha and served with bachmade ranch', '6.0', 'CORN RIBS super simple six fired corn ribs served with jalapeno & coriander crema', '5.0', 'gf,pb', '= SAUSAGES all sausages are served single with a complim

In [ ]:
#Preprocess


In [5]:
import random
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.pipeline import Pipeline, FeatureUnion

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[,;:()\-\/]', ' ', text)
    #remove weights like 300g, 200g
    text = re.sub(r"\d+[g]", "", text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

allowed = ["celery", "gluten", "crustacean", "eggs", "fish",
           "lupin", "dairy", "mollusc", "mustard", "peanuts",
            "sesame", "soy", "sulphur dioxide", "sulphites", "tree nuts"]

classes = ['celery', 'crustacean', 'dairy', 'eggs', 'fish', 'gluten', 'lupin', 'mustard',
 'peanuts', 'sesame', 'soy', 'tree nuts']

def get_allergens(text):
    food_allergens = []
    for line in text:
        allergen_array = []
        num = random.randint(0,3)
        for i in range(num):
            allergen_array.append(random.choice(allowed))
        food_allergens.append({
            "text" : line,
            "allergens" : allergen_array
        })
    return food_allergens

def predict_allergens(text):

    model = pickle.load(open('model.pickle','rb')) #pretrained classifier
    combined = pickle.load(open('combined.pickle', 'rb')) #prefitted tf-idf vectoriser
    
    food_allergens = []

    for line in text:
        line = clean_text(line)
        new_sentences = [line]
        new_sentence_tfidf = combined.transform(new_sentences)
        #probas = model.predict_proba(new_sentence_tfidf)

        predicted_sentences = model.predict(new_sentence_tfidf)
        #proba_sentence = model.predict_proba(new_sentence_tfidf)
        decoded = [classes[i] for i, val in enumerate(predicted_sentences[0]) if (val == 1).any()]
        
        food_allergens.append({
            "text" : line,
            "allergens" : decoded
        })
    
    return food_allergens




In [6]:
predict_allergens(menu_items)

[{'text': "just because we can here's some kai from our evening menu all cooked aver kanaku ~enjoy something a little different vo",
  'allergens': []},
 {'text': 'sunday specials', 'allergens': []},
 {'text': 'from kanaku', 'allergens': []},
 {'text': '12 hour smoked brisket a slice of our 12 hour smoked brisket served with bachmade potato salad and a complimentary sauce of your choice',
  'allergens': []},
 {'text': '12.0', 'allergens': []},
 {'text': '8 hour smoked beef rib tacos tender beef short rib on the bone served with six fired tacos house slaw peach salsa and bachmade jalapeno & coriander crema',
  'allergens': []},
 {'text': '23.0', 'allergens': []},
 {'text': '8 hour smoked pork belly our pork belly is cooked low and slow and just falls apart none of this undercooked shttyou often get when out in a restaurant served with apple slaw and bachmade star anise & chilli syrup',
  'allergens': []},
 {'text': 'chicken drumettes five chicken drumettes in our house rub cooked over f